# Audit — DeepSeek-V2-Lite Q8_0 rung (CPU session, no GPU quota)

**Settings:** Accelerator **None (CPU)** · Internet **On** · Secrets `HF_TOKEN` and `GITHUB_TOKEN`
(if the repo is private).
**Add Input:** model `ryzewtf/moe-panel/gguf/deepseek-v2-lite` (the panel's Q4_K_M, for the identity check).

Run All. Roughly: converter environment + tokenizer check ~1 min (fails fast if the converter would
not recognise the tokenizer), build llama-quantize ~10 min, download 31 GB ~10 min, convert to F16 ~20 min,
Q4_K_M ~15 min, Q8_0 ~10 min, identity check ~10 min. Output: `deepseek-v2-lite-Q8_0.gguf` (~16 GB)
plus `convert_report.json` in `/kaggle/working`. Then attach **this notebook's output** to
`audit_gpu_checks.ipynb`.

`convert_report.json` → `q4_identical_to_panel` must be `true` before the GPU session: it proves the
Q8_0 rung is quantized from the same F16 as the trace already collected.


In [ ]:
# ============================================================================
# PARAMETERS
# ============================================================================
GIT_REF = "VLLM_PORT"
MODEL = "deepseek-v2-lite"
PANEL_Q4_SHA = "967ab33ee0d61ebe38bd974fdc31e4c9a4eea13fd9c72bcec43d7afc9f2bbef1"   # configs/models.yaml


In [ ]:
# ============================================================================
# Bootstrap: clone the repo at GIT_REF, install what the image lacks.
# ============================================================================
import os, subprocess, sys, time, shutil
from pathlib import Path

GIT_URL = "https://github.com/ryzewtf/GenAI-IA-1.git"
REPO = Path("/kaggle/working/repo")
T0 = time.time()


def run(cmd, cwd=None, check=True, quiet=False):
    """Run a command, stream it, and fail loudly.

    `check=True` by default on purpose: a step that fails and lets the notebook carry on
    produces a session whose later cells operate on the previous step's leftovers, which is
    the expensive kind of wrong here -- it looks like it worked.
    """
    if not quiet:
        print("$", " ".join(str(c) for c in cmd), flush=True)
    p = subprocess.run([str(c) for c in cmd], cwd=cwd and str(cwd), text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if p.stdout and not quiet:
        print(p.stdout, flush=True)
    if check and p.returncode != 0:
        raise SystemExit(f"FAILED ({p.returncode}): {' '.join(str(c) for c in cmd)}")
    return p


# A private repo needs a token; a public one must not be handed one. Kaggle Secrets is the only
# place a token belongs -- an inline PAT is committed the moment the notebook is saved.
url = GIT_URL
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if tok:
        url = GIT_URL.replace("https://", f"https://{tok}@")
        print("using GITHUB_TOKEN from Kaggle Secrets")
except Exception:
    print("no GITHUB_TOKEN secret; cloning anonymously (fine if the repo is public)")

if REPO.exists():
    run(["git", "fetch", "--all", "--tags"], cwd=REPO)
    run(["git", "checkout", GIT_REF], cwd=REPO)
    run(["git", "pull", "--ff-only"], cwd=REPO, check=False)
else:
    run(["git", "clone", url, str(REPO)])
    run(["git", "checkout", GIT_REF], cwd=REPO)

HEAD = run(["git", "rev-parse", "HEAD"], cwd=REPO, quiet=True).stdout.strip()
print(f"\nrepo at {HEAD}  (ref {GIT_REF})")

os.chdir(REPO)
sys.path.insert(0, str(REPO))

# --no-deps: the Kaggle image pins torch/numpy against each other, and letting pip resolve
# transitive deps here is how a session ends up with a numpy the image's torch was not built
# against. Same rule as PIP_PACKAGES in src/runtime/setup_kaggle.py.
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
     "huggingface_hub", "datasets", "sentencepiece", "gguf", "kagglehub"], check=False)

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")   # xet hangs at 0%/99% in managed notebooks
os.environ.setdefault("PYTHONUNBUFFERED", "1")

# The multilingual domain's parallel control (facebook/flores) and the code substitute
# (starcoderdata) are gated on the HF Hub: without a token the corpus fetch dies with
# DatasetNotFoundError partway through. The token lives in Kaggle Secrets, never inline --
# an HF_TOKEN pasted into a cell is leaked the moment the notebook is saved. Both env names
# are set because huggingface_hub reads HF_TOKEN and older datasets code reads the other.
try:
    from kaggle_secrets import UserSecretsClient
    _hf = UserSecretsClient().get_secret("HF_TOKEN")
    if _hf:
        os.environ["HF_TOKEN"] = _hf
        os.environ["HUGGINGFACE_HUB_TOKEN"] = _hf
        print("using HF_TOKEN from Kaggle Secrets")
except Exception:
    print("no HF_TOKEN secret; gated datasets (flores, starcoderdata) will 404 in a corpus run")


In [ ]:
# ============================================================================
# Locate the attached panel Q4_K_M and check disk BEFORE the slow steps.
# ============================================================================
import json as _json, shutil as _sh
hits = sorted(Path("/kaggle/input").rglob("deepseek-v2-lite-Q4_K_M.gguf"))
if len(hits) != 1:
    raise SystemExit(f"attach ryzewtf/moe-panel/gguf/deepseek-v2-lite (found {hits})")
PANEL_Q4 = hits[0]
for d in ("/kaggle/temp", "/tmp", "/kaggle/working"):
    if Path(d).is_dir():
        print(f"{d:16s} {_sh.disk_usage(d).free/2**30:7.1f} GiB free")
print("panel Q4_K_M", PANEL_Q4, f"{PANEL_Q4.stat().st_size/2**30:.1f} GiB")


In [ ]:
# ============================================================================
# Converter environment. The converter names the pre-tokenizer from a hash of the tokenizer's
# output, and that output depends on the transformers version: 4.57.6 (the panel conversion,
# results/convert_deepseek-v2-lite.json) gives 049ecf76..., which the pinned tree knows; the image's
# transformers 5.x gives 99ecacb4..., which it does not, and the converter stops ("BPE pre-tokenizer
# was not recognized"). The first run of this notebook died there, after the 31 GB download.
# So the converter runs from the image's interpreter with the recorded transformers/tokenizers
# shadowing the image's through PYTHONPATH. --no-deps leaves the image's numpy and torch alone.
# The hash is checked here, before anything slow.
# ============================================================================
CONV_SITE = Path("/tmp/convsite")
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--target", str(CONV_SITE),
     "transformers==4.57.6", "tokenizers==0.22.2", "huggingface_hub==0.36.0"])
CONV_PY = Path("/tmp/convpy")
CONV_PY.write_text(f'#!/bin/sh\nPYTHONPATH={CONV_SITE}${{PYTHONPATH:+:$PYTHONPATH}} exec {sys.executable} "$@"\n')
CONV_PY.chmod(0o755)

from huggingface_hub import snapshot_download
TOK_DIR = snapshot_download("deepseek-ai/DeepSeek-V2-Lite", local_dir="/tmp/ds_tok",
                            allow_patterns=["tokenizer*.json", "config.json"])
EXPECTED_CHKHSH = "049ecf7629871e3041641907f3de7c733e4dbfdc736f57d882ba0b0845599754"
CHKTXT_FILE = Path("/tmp/chktxt.txt")   # a file, not argv: the text is multi-line and full of quotes
CHKTXT_FILE.write_text('\n \n\n \n\n\n \t \t\t \t\n  \n   \n    \n     \n\U0001f680 (normal) \U0001f636\u200d\U0001f32b\ufe0f (multiple emojis concatenated) \u2705 \U0001f999\U0001f999 3 33 333 3333 33333 333333 3333333 33333333 3.3 3..3 3...3 \u1780\u17b6\u1793\u17cb\u178f\u17c2\u1796\u17b7\u179f\u17c1\u179f\u17a2\u17b6\u1785\U0001f601 ?\u6211\u60f3\u5728apple\u5de5\u4f5c1314151\u5929\uff5e ------======= \u043d\u0435\u0449\u043e \u043d\u0430 \u0411\u044a\u043b\u0433\u0430\u0440\u0441\u043a\u0438 \'\'\'\'\'\'```````""""......!!!!!!?????? I\'ve been \'told he\'s there, \'RE you sure? \'M not sure I\'ll make it, \'D you like some tea? We\'Ve a\'lL', encoding="utf-8", newline="")
probe = run([str(CONV_PY), "-c",
             "import sys, transformers, tokenizers; from hashlib import sha256; "
             "from transformers import AutoTokenizer; "
             "t = AutoTokenizer.from_pretrained(sys.argv[1]); "
             "s = open(sys.argv[2], encoding='utf-8', newline='').read(); "
             "print(transformers.__version__, tokenizers.__version__, "
             "sha256(str(t.encode(s)).encode()).hexdigest())",
             TOK_DIR, str(CHKTXT_FILE)], quiet=True, check=False)
got = probe.stdout.strip().splitlines()[-1].split() if probe.stdout.strip() else []
print("converter env:", " ".join(got) or probe.stdout)
if probe.returncode != 0 or got[:2] != ["4.57.6", "0.22.2"] or got[2:] != [EXPECTED_CHKHSH]:
    raise SystemExit("converter environment is not the recorded one; nothing was downloaded")


In [ ]:
# ============================================================================
# The panel recipe: HF safetensors -> F16 -> Q4_K_M at the pinned llama.cpp commit (CPU build).
# --keep-f16 keeps the F16 for the Q8_0 rung; --prune drops the safetensors once converted.
# --python: the converter runs under the environment checked above.
# ============================================================================
run([sys.executable, "scripts/kaggle_convert.py", "--model", MODEL, "--keep-f16", MODEL, "--prune",
     "--python", str(CONV_PY)])
from src.runtime.setup_kaggle import _pick_scratch
GGUF_DIR = _pick_scratch() / "gguf"
F16 = GGUF_DIR / f"{MODEL}-F16.gguf"
Q4_NEW = GGUF_DIR / f"{MODEL}-Q4_K_M.gguf"
for p in (F16, Q4_NEW):
    if not p.is_file():
        raise SystemExit(f"missing {p}; see the conversion log above")
    print(p, f"{p.stat().st_size/2**30:.1f} GiB")


In [ ]:
# ============================================================================
# Q8_0 from the same F16, then the T1.2 router-dtype audit on it.
# ============================================================================
QUANTIZE = next(p for root in ("/tmp", "/kaggle/temp") for p in Path(root).rglob("llama-quantize")
                if p.is_file() and "CMakeFiles" not in str(p))
Q8 = Path("/kaggle/working") / f"{MODEL}-Q8_0.gguf"
if run([str(QUANTIZE), str(F16), str(Q8), "Q8_0", str(os.cpu_count() or 4)], check=False).returncode != 0:
    Q8.unlink(missing_ok=True)
    raise SystemExit("llama-quantize Q8_0 failed")
audit = run([sys.executable, "-m", "src.capture.router_audit", str(Q8),
             "-o", f"/kaggle/working/router_dtype_audit_{Q8.stem}.csv"], check=False)
ROUTER_OK = audit.returncode == 0
print("Q8_0", Q8, f"{Q8.stat().st_size/2**30:.1f} GiB", "router audit", "PASS" if ROUTER_OK else "FAIL")


In [ ]:
# ============================================================================
# Identity check: the regenerated Q4_K_M against the panel's, file hash and per-tensor data hash.
# A file-hash mismatch alone can be metadata (e.g. a converter-env string); identical tensor data is
# what makes the Q8_0 rung a same-source comparison.
# ============================================================================
import hashlib
from gguf import GGUFReader

def file_sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 24), b""):
            h.update(b)
    return h.hexdigest()

def tensor_hashes(p):
    r = GGUFReader(str(p))
    import numpy as np
    return {t.name: (str(t.tensor_type), hashlib.sha256(np.ascontiguousarray(t.data)).hexdigest())
            for t in r.tensors}

new_sha, panel_sha = file_sha(Q4_NEW), file_sha(PANEL_Q4)
a, b = tensor_hashes(Q4_NEW), tensor_hashes(PANEL_Q4)
differ = sorted(k for k in set(a) | set(b) if a.get(k) != b.get(k))
report = {
    "model": MODEL, "q8_file": Q8.name, "q8_sha256": file_sha(Q8), "q8_size_bytes": Q8.stat().st_size,
    "q8_router_audit_pass": ROUTER_OK,
    "q4_regenerated_sha256": new_sha, "q4_panel_sha256": panel_sha,
    "q4_panel_sha_matches_models_yaml": panel_sha == PANEL_Q4_SHA,
    "q4_file_identical": new_sha == panel_sha,
    "q4_n_tensors": len(a), "q4_tensors_differing": differ[:50], "q4_n_tensors_differing": len(differ),
    "q4_identical_to_panel": not differ and set(a) == set(b),
}
Path("/kaggle/working/convert_report.json").write_text(_json.dumps(report, indent=2))
for rec in Path("results").rglob("*conversion*.json"):
    shutil.copy(rec, "/kaggle/working/")
print(_json.dumps(report, indent=2))
if not report["q4_identical_to_panel"] or not ROUTER_OK:
    print("\n!! DO NOT start the GPU session on this output: see the report above")
# The clone is not output. Left in /kaggle/working it is attached to the GPU notebook along with the
# Q8_0 (where `_one(...)` must find exactly one of each input), counts against the output cap, and
# makes `kaggle kernels output` page through thousands of files.
os.chdir("/kaggle/working")
shutil.rmtree(REPO, ignore_errors=True)
print(sorted(p.name for p in Path("/kaggle/working").iterdir()))
print(f"\ndone in {(time.time()-T0)/60:.1f} min")
